# Lab Activity 3: k-Nearest Neighbors


**Learning objective:** Follow the exact lecture procedure: (1) choose k, (2) calculate euclidean distances, (3) sort distances, (4) select neighbors, (5) majority vote, (6) predict.


## Part 1 — Euclidean distance and k-NN classification

**Training Data**:

1.   A=(1,1) Red
2.   B=(2,2) Red
3.   C=(4,3) Blue
4.   D=(5,5) Blue

**Testing Data**:

1.   P=(3,3)



In [32]:
from math import sqrt

training = [
    {"Point":"A", "X":1, "Y":1, "Class":"Red"},
    {"Point":"B", "X":2, "Y":2, "Class":"Red"},
    {"Point":"C", "X":4, "Y":3, "Class":"Blue"},
    {"Point":"D", "X":5, "Y":4, "Class":"Blue"},
]
test_x, test_y =4,4

# Step 1: choose k
k =3
print('STEP 1: k =', k)

# Step 2: compute distances using the formula in the lecture
rows = []
for item in training:
    dx = test_x - item['X']
    dy = test_y - item['Y']
    distance = sqrt(dx**2 + dy**2)
    rows.append({'Point':item['Point'], 'Class':item['Class'], 'Distance':distance})
    print(f"STEP 2: d(P,{item['Point']}) = sqrt(({test_x}-{item['X']})² + ({test_y}-{item['Y']})²) = {distance:.3f}")

# Step 3: sort from nearest to farthest
ordered = sorted(rows, key=lambda row: row['Distance'])
print('Step 3: sorted:', [(r['Point'], round(r['Distance'], 3)) for r in ordered])


# Step 4: select the k nearest neighbors
neighbors = ordered[:k]
print('Step 4: selected:', [(r['Point'], r['Class'], 3) for r in neighbors])

# Step 5: count votes manually
votes = {}
for item in neighbors:
    label = item['Class']
    votes[label] = votes.get(label, 0) + 1
print('STEP 5: votes:', votes)

# Step 6: class with the most votes
prediction = max(votes, key=votes.get)
print('STEP 6: predicted class=', prediction)


STEP 1: k = 3
STEP 2: d(P,A) = sqrt((4-1)² + (4-1)²) = 4.243
STEP 2: d(P,B) = sqrt((4-2)² + (4-2)²) = 2.828
STEP 2: d(P,C) = sqrt((4-4)² + (4-3)²) = 1.000
STEP 2: d(P,D) = sqrt((4-5)² + (4-4)²) = 1.000
Step 3: sorted: [('C', 1.0), ('D', 1.0), ('B', 2.828), ('A', 4.243)]
Step 4: selected: [('C', 'Blue', 3), ('D', 'Blue', 3), ('B', 'Red', 3)]
STEP 5: votes: {'Blue': 2, 'Red': 1}
STEP 6: predicted class= Blue


## Part 2 — Manual min-max normalization
Use the *age/salary* data from the slides. Normalize each feature using the **training-set** minimum and maximum: `normalized = (value - min)/(max - min)`. Then calculate all four distances and vote for k=1 and k=3.

In [33]:
people = [
    {'Person':'A','Age':25,'Salary':30000,'Class':'Red'},
    {'Person':'B','Age':30,'Salary':35000,'Class':'Red'},
    {'Person':'C','Age':45,'Salary':80000,'Class':'Blue'},
    {'Person':'D','Age':50,'Salary':90000,'Class':'Blue'},
]
test = {'Person':'P','Age':35,'Salary':50000}
min_age = min(p['Age'] for p in people)
max_age = max(p['Age'] for p in people)
min_salary = min(p['Salary'] for p in people)
max_salary = max(p['Salary'] for p in people)
print('Training Age min/max:', min_age, max_age)
print('Training Salary min/max:', min_salary, max_salary)

def normalize(value, minimum, maximum):
    return (value - minimum) / (maximum - minimum)

normalized = []
for p in people:
    age = normalize(p['Age'], min_age, max_age)
    salary = normalize(p['Salary'], min_salary, max_salary)
    normalized.append({'Person':p['Person'],'Age':age,'Salary':salary,'Class':p['Class']})
    print(f"{p['Person']}: age=({p['Age']}-{min_age})/({max_age}-{min_age})={age:.3f}; salary=({p['Salary']}-{min_salary})/({max_salary}-{min_salary})={salary:.3f}")

p_age = normalize(test['Age'], min_age, max_age)
p_salary = normalize(test['Salary'], min_salary, max_salary)
print(f'P normalized: ({p_age:.3f}, {p_salary:.3f})')

# Steps 2 and 3: distances and sorting
age_distances = []
for p in normalized:
    d = sqrt((p_age-p['Age'])**2 + (p_salary-p['Salary'])**2)
    age_distances.append({'Person':p['Person'],'Class':p['Class'],'Distance':d})
    print(f"d(P,{p['Person']}) = sqrt(({p_age:.3f}-{p['Age']:.3f})² + ({p_salary:.3f}-{p['Salary']:.3f})²) = {d:.3f}")
age_distances = sorted(age_distances, key=lambda r:r['Distance'])
print('Nearest first:', [(r['Person'],round(r['Distance'],3)) for r in age_distances])

for k in [1,2,3]:
    nearest = age_distances[:k]
    votes = {}
    for r in nearest:
        votes[r['Class']] = votes.get(r['Class'],0)+1
    print(f'k={k}; selected={[(r['Person'], r['Class']) for r in nearest]}; votes={votes}; prediction={max(votes,key=votes.get)}')


Training Age min/max: 25 50
Training Salary min/max: 30000 90000
A: age=(25-25)/(50-25)=0.000; salary=(30000-30000)/(90000-30000)=0.000
B: age=(30-25)/(50-25)=0.200; salary=(35000-30000)/(90000-30000)=0.083
C: age=(45-25)/(50-25)=0.800; salary=(80000-30000)/(90000-30000)=0.833
D: age=(50-25)/(50-25)=1.000; salary=(90000-30000)/(90000-30000)=1.000
P normalized: (0.400, 0.333)
d(P,A) = sqrt((0.400-0.000)² + (0.333-0.000)²) = 0.521
d(P,B) = sqrt((0.400-0.200)² + (0.333-0.083)²) = 0.320
d(P,C) = sqrt((0.400-0.800)² + (0.333-0.833)²) = 0.640
d(P,D) = sqrt((0.400-1.000)² + (0.333-1.000)²) = 0.897
Nearest first: [('B', 0.32), ('A', 0.521), ('C', 0.64), ('D', 0.897)]
k=1; selected=[('B', 'Red')]; votes={'Red': 1}; prediction=Red
k=2; selected=[('B', 'Red'), ('A', 'Red')]; votes={'Red': 2}; prediction=Red
k=3; selected=[('B', 'Red'), ('A', 'Red'), ('C', 'Blue')]; votes={'Red': 2, 'Blue': 1}; prediction=Red


## Case 1 — Numeric prediction
For numerical output, use the average of the **k** nearest outputs (not majority voting). A: x=1→10, B: x=2→20, C: x=4→40, D: x=5→50. Predict at E: x=3 with k=2.

In [34]:
numeric = [('A',1,10),('B',2,20),('C',4,40),('D',5,50)]
test_x =4.5
k =2
numeric_distances = []
for point,x,value in numeric:
    d = sqrt((test_x-x)**2)
    numeric_distances.append((point,d,value))
    print(f'd(E,{point}) = sqrt(({test_x}-{x})²) = {d:.3f}')
numeric_distances.sort(key=lambda r:r[1])
selected = numeric_distances[:k]
print('Sorted:', numeric_distances)
print('Nearest:', selected)
predicted_value = sum(r[2] for r in selected) / k
print('Predicted numerical output =',predicted_value)


d(E,A) = sqrt((4.5-1)²) = 3.500
d(E,B) = sqrt((4.5-2)²) = 2.500
d(E,C) = sqrt((4.5-4)²) = 0.500
d(E,D) = sqrt((4.5-5)²) = 0.500
Sorted: [('C', 0.5, 40), ('D', 0.5, 50), ('B', 2.5, 20), ('A', 3.5, 10)]
Nearest: [('C', 0.5, 40), ('D', 0.5, 50)]
Predicted numerical output = 45.0


## Case 2 — Non-numeric attributes
For each attribute, **same = 0**, **different = 1**. Sum the attribute differences as in the lecture example. Test P=(Blue, Circle), k=3. Note: a tie at the boundary can occur; here A is distance 0, C and D distance 1, so the three nearest are unambiguous.

In [35]:
categorical = [
    ('A','Blue','Circle','Yes'),
    ('B','Red','Square','No'),
    ('C','Blue','Square','Yes'),
    ('D','Green','Circle','No'),
]
test_color, test_shape = 'Red','Square'
k = 2
cat_distances = []
for person,color,shape,label in categorical:
    color_difference = 0 if color == test_color else 1
    shape_difference = 0 if shape == test_shape else 1
    distance = color_difference + shape_difference
    cat_distances.append((person,distance,label))
    print(f'{person}: color difference={color_difference}; shape difference={shape_difference}; total={distance}')
cat_distances.sort(key=lambda r:r[1])
nearest = cat_distances[:k]
votes = {}
for _,_,label in nearest:
    votes[label] = votes.get(label,0)+1
print('Sorted:',cat_distances)
print('Selected:',nearest)
print('Votes:',votes)
print('Prediction:',max(votes,key=votes.get))


A: color difference=1; shape difference=1; total=2
B: color difference=0; shape difference=0; total=0
C: color difference=1; shape difference=0; total=1
D: color difference=1; shape difference=1; total=2
Sorted: [('B', 0, 'No'), ('C', 1, 'Yes'), ('A', 2, 'Yes'), ('D', 2, 'No')]
Selected: [('B', 0, 'No'), ('C', 1, 'Yes')]
Votes: {'No': 1, 'Yes': 1}
Prediction: No


## Case 3 — Missing values
**Lecture rule:** When a normalized attribute is missing, use the maximum attribute difference **1**. Then compute Euclidean distance. P=(Age 0.6, Salary 0.5). B has missing salary. This is the special rule taught in the slides, not the only possible missing-data strategy.

In [36]:
missing_data = [
    ('A',0.2,0.3,'Yes'),
    ('B',0.5,None,'No'),
    ('C',0.8,0.7,'Yes'),
    ('D',0.9,0.6,'No'),
]
test_age, test_salary = 0.7,0.6
missing_distances = []
for person,age,salary,label in missing_data:
    age_difference = abs(test_age-age)
    salary_difference = 1 if salary is None else abs(test_salary-salary)
    distance = sqrt(age_difference**2 + salary_difference**2)
    missing_distances.append((person,distance,label))
    print(f'{person}: sqrt(({age_difference:.3f})² + ({salary_difference:.3f})²) = {distance:.3f}')
missing_distances.sort(key=lambda r:r[1])
print('Sorted nearest first:',missing_distances)
for k in [1,2,3]:
    nearest = missing_distances[:k]
    votes = {}
    for _,_,label in nearest:
        votes[label] = votes.get(label,0)+1
    print(f'k={k}, nearest={nearest}, votes={votes}, prediction={max(votes,key=votes.get)}')


A: sqrt((0.500)² + (0.300)²) = 0.583
B: sqrt((0.200)² + (1.000)²) = 1.020
C: sqrt((0.100)² + (0.100)²) = 0.141
D: sqrt((0.200)² + (0.000)²) = 0.200
Sorted nearest first: [('C', 0.14142135623730953, 'Yes'), ('D', 0.20000000000000007, 'No'), ('A', 0.58309518948453, 'Yes'), ('B', 1.019803902718557, 'No')]
k=1, nearest=[('C', 0.14142135623730953, 'Yes')], votes={'Yes': 1}, prediction=Yes
k=2, nearest=[('C', 0.14142135623730953, 'Yes'), ('D', 0.20000000000000007, 'No')], votes={'Yes': 1, 'No': 1}, prediction=Yes
k=3, nearest=[('C', 0.14142135623730953, 'Yes'), ('D', 0.20000000000000007, 'No'), ('A', 0.58309518948453, 'Yes')], votes={'Yes': 2, 'No': 1}, prediction=Yes


## Part 6 — Selecting k
There is **no fixed best k**. The lecture recommends evaluating several k values and selecting one with low error on held-out labeled examples. With only four training rows, use k=1 and k=3 for illustration; a larger dataset is needed for meaningful model selection.

In [37]:
# Explore the age/salary example with the already computed sorted distances.
for k in [1,2,3]:
    neighbors = age_distances[:k]
    counts = {}
    for r in neighbors:
        counts[r['Class']] = counts.get(r['Class'],0)+1
    print('k =',k,'| neighbors =',[r['Person'] for r in neighbors], '| votes =',counts,'| prediction =',max(counts,key=counts.get))


k = 1 | neighbors = ['B'] | votes = {'Red': 1} | prediction = Red
k = 2 | neighbors = ['B', 'A'] | votes = {'Red': 2} | prediction = Red
k = 3 | neighbors = ['B', 'A', 'C'] | votes = {'Red': 2, 'Blue': 1} | prediction = Red
